In [ ]:
""" 
1.IC(information coefficient):信息系数，
（1）：截面维度：（每期：每月，每天等等），用当期因子的暴露值，和下期资产收益率的皮尔逊相关系数，
其本质在于，当期因子能不能用来预测未来收益
IC - [-1,1],IC>0 正向相关因子，因子越大，未来收益越高（动量，趋势，质量）
IC < 0 负向相关因子，因子越小，未来收益越高（短期反转，低波动，价值）
IC近似为0，没有预测能力，完全失效

一般ic >= 0.02就属于有效因子了

2.Information Ratio（IR）信息比率
well，因子IR和组合超额收益率IR是不一样的，注意区分
ir = ic.mean()/ic.std()
ir 越高，因子越稳定

ir >= 0.5,合格因子，
ir >= 0.75 优秀因子
ir >= 1.0 顶级核心因子


3.多因子进行组合加权的情况，
well，在进行因子组合的时候，首先要对因子进行标准化处理（Z-score）
1.equal weighted 等权组合，其底层逻辑为假设每个因子的预测能力相等
2.ICIR weighted（ICIR组合加权）：其逻辑为近期的表现越好的因子，给予更高的权重，wi = iri / 长期加权求和的iri

4.在现实生活中，寻找alpha其实已经是非常苦难和不可能实现的事情了，现在的风向更多是来寻找beta，以及处理好拥挤度和因子
之间的关系
"""

In [ ]:
""" 
斯皮尔曼秩相关系数（）：
需要导入相关的库，from scipy.stats import spearmanr
.intersection:取股票交集，（双重保险对齐，）

df.loc[行索引，列索引 ]两个都取表示精准定位

spearmanr 返回的是两个值，一个是ic，另一个是p值，（前边是ic

.describe(),返回八行，这八行都是统计指标，并且是固定的，但是后面也可以自定义参数
其接收的类型是DataFrame或者Series，然后返还的是DataFrame
.div(...,axis = 0)表示除以当天的标准差，归一化
"""

In [ ]:
def calculate_ic(self, factor: pd.DataFrame, 
                     forward_returns: Optional[pd.DataFrame] = None) -> pd.Series:
        """
            计算因子的 Rank IC 序列（Spearman 相关性）
    
            Parameters:
            factor: 因子值，行=日期，列=股票
            forward_returns: 下期对数收益率，默认用 self.returns.shift(-1)
    
            Returns:
            Series: 每日 IC 值
        """
        if forward_returns is None:
            # self.returns 已是对数收益率（连续复利）
            forward_returns = self.returns.shift(-1)
    
        ic_values = {}
    
        for date in factor.index[:-1]:  # 最后一天没有下期收益
            f = factor.loc[date].dropna()
            if len(f) < 10:
                continue
        
            # 找到下一个交易日
            next_dates = forward_returns.index[forward_returns.index > date]
            if len(next_dates) == 0:
                    break
            next_date = next_dates[0]
        
            r = forward_returns.loc[next_date].dropna()
        
        # 取交集
            common = f.index.intersection(r.index)
            if len(common) >= 10:
                ic, _ = spearmanr(f[common], r[common])
                ic_values[date] = ic
    
        return pd.Series(ic_values, name='IC')


    def calculate_ic_summary(self, factor: pd.DataFrame) -> Dict:
        """计算因子 IC 摘要统计"""
        ic_series = self.calculate_ic(factor)
    
        n = len(ic_series)
        ic_mean = ic_series.mean()
        ic_std = ic_series.std(ddof=0)  # 总体标准差（与波动率计算一致）
    
        return {
            'ic_mean': ic_mean,
            'ic_std': ic_std,
            'ic_ir': ic_mean / ic_std if ic_std > 0 else 0,  # Information Ratio
            'ic_win_rate': (ic_series > 0).mean(),           # IC>0 的比例
            'ic_t_stat': ic_mean / ic_std * np.sqrt(n) if ic_std > 0 else 0,  # t统计量
            'ic_positive_days': (ic_series > 0).sum(),
            'ic_negative_days': (ic_series < 0).sum(),
            'n_obs': n
        }


    def combine_factors_equal_weight(self, factors: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """
        等权组合多个因子（横截面标准化后加总）
        
        Parameters:
        factors: 字典 {因子名: 因子值DataFrame}
    
        Returns:
        DataFrame: 组合因子值
        """
        # 初始化（用第一个因子的结构）
        first_factor = list(factors.values())[0]
        combined = pd.DataFrame(0.0, index=first_factor.index, columns=first_factor.columns)
    
        for name, factor in factors.items():
            # 横截面标准化：每天的所有股票中，均值=0，标准差=1
            # axis=1 表示对每行（每个日期）计算跨股票的统计量
            zscore = factor.sub(factor.mean(axis=1), axis=0).div(factor.std(axis=1), axis=0)
            combined = combined + zscore.fillna(0)  # 处理 NaN
    
        return combined / len(factors)